### Link length optimization

In [11]:
import numpy as np
import kinematics_utils as ku
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.optimize import differential_evolution

In [12]:
def mechanical_advantage(l_fixed, l_crank, l_rod, crank_angle_deg ,pitch_deg, roll_deg):
    
    l_diff = l_rod - l_fixed
    l_spacing_rod=80.0
    l_spacing_crank=80.0
    l_spacing_base=80.0
    rod_angle_deg=90.0
    crank_angle = np.radians(crank_angle_deg)
    rod_angle   = np.radians(rod_angle_deg)
    pitch       = np.radians(pitch_deg)
    roll        = np.radians(roll_deg)

    # Home positions
    rA1_H = np.array([0.0, +l_spacing_crank / 2.0, l_diff + l_crank * np.sin(crank_angle)])
    rA2_H = np.array([0.0, -l_spacing_crank / 2.0, l_diff + l_crank * np.sin(crank_angle)])
    rB1_H = np.array([l_crank * np.cos(crank_angle), +l_spacing_rod / 2.0, l_diff])
    rB2_H = np.array([l_crank * np.cos(crank_angle), -l_spacing_rod / 2.0, l_diff])
    rC1_H = np.array([l_crank * np.cos(crank_angle) + l_rod * np.cos(rod_angle),+l_spacing_base / 2.0,-l_fixed,])
    rC2_H = np.array([l_crank * np.cos(crank_angle) + l_rod * np.cos(rod_angle),-l_spacing_base / 2.0,-l_fixed,])


    R = ku.Ry(pitch) @ ku.Rx(roll)

    rA1 = R @ rA1_H
    rA2 = R @ rA2_H
    rB1_i = R @ rB1_H
    rB2_i = R @ rB2_H

    axis = rA1 - rA2
    axis /= np.linalg.norm(axis)

    # IK
    theta1_candidates = ku.solve_theta(rC1_H, rA1, rB1_i, l_rod, axis)
    theta2_candidates = ku.solve_theta(rC2_H, rA2, rB2_i, l_rod, axis)

    theta1 = theta1_candidates[1]
    theta2 = theta2_candidates[1]

    MA_pitch = np.abs(theta1 / pitch) if pitch_deg != 0 else None
    return MA_pitch


In [13]:
def objective(design_variables):

    l_fixed, l_crank, l_rod, crank_angle_deg = design_variables

    MA = mechanical_advantage(
        l_fixed=l_fixed,
        l_crank=l_crank,
        l_rod=l_rod,
        crank_angle_deg=crank_angle_deg,
        pitch_deg=15.0,
        roll_deg=0.0,
    )

    if MA is None or np.isnan(MA):
        return 1e6  # infeasible - penalty
    return -MA  # negative for maximization


constraints = [
    (0, 25),  # l_fixed
    (70, 100),  # l_crank
    (100, 120),  # l_rod
    (0, 0),  # crank_angle_deg
]

initial_guess = [0, 0, 0, 0]

#### Sequential Least Squares Quadratic Programming

In [14]:
result = minimize(
    objective,
    initial_guess,
    method="SLSQP",
    bounds=constraints,
    options={"disp": False, "ftol": 1e-8},
)

#### L-BFGS-B (Limited-memory Broyden–Fletcher–Goldfarb–Shanno with Bounds)

In [15]:

# result = minimize(objective, initial_guess, method='L-BFGS-B', bounds=constraints)

#### Differential Evolution

In [16]:
# result = differential_evolution( objective, bounds=constraints,strategy='best1bin',maxiter=1000,
#                                 popsize=15, tol=1e-6, polish=True, disp=False )

In [17]:
if result.success:
    best_l_fixed, best_l_crank, best_l_rod, best_crank_angle_deg = result.x
    max_MA = -result.fun
    print("Optimization successful!\n")
    print(f"Optimal link parameters:")
    print(f"  l_fixed          = {best_l_fixed:.3f}")
    print(f"  l_crank          = {best_l_crank:.3f}")
    print(f"  l_rod            = {best_l_rod:.3f}")
    print(f"  crank_angle_deg  = {best_crank_angle_deg:.3f}\n")
    print(f"Maximum Mechanical Advantage = {max_MA:.3f}")
else:
    print("Optimization failed:", result.message)

Optimization successful!

Optimal link parameters:
  l_fixed          = 25.000
  l_crank          = 70.000
  l_rod            = 120.000
  crank_angle_deg  = 0.000

Maximum Mechanical Advantage = 1.038
